# KRONOS2 spatial characterisation


In [ ]:
from __future__ import annotations
from pathlib import Path
import os
import sys

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / "config" / "project.yaml").is_file()
             and (path / "src" / "pathology").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter within this repository.")
sys.path.insert(0, str(ROOT / "src"))
from pathology.config import load_paths
PATHS = load_paths(ROOT)
os.environ.setdefault("OMP_NUM_THREADS", "8")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "8")
os.environ.setdefault("HF_HOME", str(ROOT / ".cache" / "huggingface"))
os.environ.setdefault("HF_MODULES_CACHE", str(ROOT / ".cache" / "huggingface" / "modules"))
os.environ.setdefault("TORCH_HOME", str(ROOT / ".cache" / "torch"))


## Configuration


In [ ]:
from pathlib import Path
from datetime import datetime
from itertools import combinations
import json
import os
import platform
import sys
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import patches as mpl_patches
from scipy import stats
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform
from sklearn.cluster import AgglomerativeClustering, MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, fowlkes_mallows_score, silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
from IPython.display import display
ANALYSIS_VERSION = '2.0'
RESULTS = PATHS.run
CASES = RESULTS / 'cases'
FEATURE_DATA = RESULTS / 'feature_characterisation' / 'feature_statistics' / 'data'
FORCE = os.getenv('KRONOS2_FORCE', '0') == '1'
QUICK_TEST = os.getenv('KRONOS2_QUICK_TEST', '0') == '1'
SEED = 6211
PCA_DIM = 12 if QUICK_TEST else 32
FINE_K_CANDIDATES = [6, 8] if QUICK_TEST else [6, 8, 10, 12, 15, 18]
COARSE_K_CANDIDATES = [2, 3] if QUICK_TEST else [2, 3, 4, 5]
PATCH_STABILITY_RUNS = 4 if QUICK_TEST else 20
SPATIAL_PERMUTATIONS = 49 if QUICK_TEST else 999
PATIENT_BOOTSTRAPS = 20 if QUICK_TEST else 250
PCR_PERMUTATIONS = 199 if QUICK_TEST else 4999
GRAPH_STEPS = 2
GRAPH_ALPHA = 0.5
MIN_PATCH_CLUSTER_FRACTION = 0.02
MIN_PATIENT_GROUP_SIZE = 3 if QUICK_TEST else 5
default_output = RESULTS / 'feature_characterisation' / ('spatial_community_graph_quick_test_v2' if QUICK_TEST else 'spatial_community_graph')
OUTPUT = Path(os.getenv('KRONOS2_COMMUNITY_OUTPUT', str(default_output)))
DATA = OUTPUT / 'data'
FIGURES = OUTPUT / 'figures'
CHECKPOINTS = OUTPUT / 'checkpoints'
for directory in [DATA, FIGURES, CHECKPOINTS]:
    directory.mkdir(parents=True, exist_ok=True)
CONFIG = {'analysis_version': ANALYSIS_VERSION, 'quick_test': QUICK_TEST, 'seed': SEED, 'pca_dim': PCA_DIM, 'fine_k_candidates': FINE_K_CANDIDATES, 'coarse_k_candidates': COARSE_K_CANDIDATES, 'patch_stability_runs': PATCH_STABILITY_RUNS, 'spatial_permutations': SPATIAL_PERMUTATIONS, 'patient_bootstraps': PATIENT_BOOTSTRAPS, 'pcr_permutations': PCR_PERMUTATIONS, 'graph_steps': GRAPH_STEPS, 'graph_alpha': GRAPH_ALPHA}
required = [FEATURE_DATA / 'case_level_features_final.parquet', FEATURE_DATA / 'patch_index_final.parquet']
missing = [str(path) for path in required if not path.exists()]
assert not missing, f'Missing inputs: {missing}'
PALETTE = ['#58508D', '#FF6361', '#FFA600', '#2F9C95', '#4C78A8', '#B279A2', '#72B7B2', '#E45756', '#54A24B', '#F2CF5B', '#8C6D31', '#6B8E23', '#A05195', '#003F5C', '#BC5090', '#7A5195', '#EF5675', '#374C80']
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 300, 'font.family': 'DejaVu Sans', 'axes.titleweight': 'bold', 'axes.spines.top': False, 'axes.spines.right': False})
labels_qc = pd.read_parquet(FEATURE_DATA / 'case_level_features_final.parquet', columns=['pcr_label'])
label_counts = labels_qc.pcr_label.value_counts()
if min(label_counts.get(0, 0), label_counts.get(1, 0)) < 2:
    raise ValueError("The report's outcome analyses require at least two labelled cases in each pCR group.")


## Statistical utilities


In [ ]:
META_COLUMNS = ['case_id', 'wsi_name', 'registration_status', 'roi_id', 'roi_x', 'roi_y', 'roi_width', 'roi_height', 'patch_id', 'patch_x', 'patch_y', 'patch_size', 'patch_stride', 'he_full_x', 'he_full_y', 'segmentation_mpp', 'tissue_fraction', 'tumour_fraction', 'tumour_associated_stroma_fraction', 'inflamed_stroma_fraction', 'other_fraction', 'class_entropy', 'dominant_class', 'mif_valid_fraction', 'mif_signal_fraction', 'extraction_status']
COMPOSITION_COLUMNS = ['tissue_fraction', 'tumour_fraction', 'tumour_associated_stroma_fraction', 'inflamed_stroma_fraction', 'other_fraction', 'class_entropy']
TISSUE_COLUMNS = ['tumour_fraction', 'tumour_associated_stroma_fraction', 'inflamed_stroma_fraction', 'other_fraction']

def bh_fdr(p_values):
    p = np.asarray(p_values, dtype=float)
    q = np.full(p.shape, np.nan)
    valid = np.isfinite(p)
    values = p[valid]
    if values.size == 0:
        return q
    order = np.argsort(values)
    ranked = values[order]
    adjusted = ranked * values.size / np.arange(1, values.size + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    restored = np.empty(values.size)
    restored[order] = np.minimum(adjusted, 1.0)
    q[valid] = restored
    return q

def normalized_entropy(counts):
    counts = np.asarray(counts, dtype=float)
    probabilities = counts[counts > 0] / counts.sum()
    if probabilities.size <= 1:
        return 0.0
    return float(stats.entropy(probabilities) / np.log(len(counts)))

def spatial_pairs(frame):
    source, target = ([], [])
    roi_groups = {}
    for key, indices in frame.groupby(['case_id', 'roi_id'], sort=False).groups.items():
        idx = np.asarray(list(indices), dtype=int)
        roi_groups[key] = idx
        group = frame.loc[idx]
        stride = int(group['patch_stride'].mode().iloc[0])
        lookup = {(int(x), int(y)): int(i) for i, x, y in zip(idx, group['patch_x'], group['patch_y'])}
        for i, x, y in zip(idx, group['patch_x'], group['patch_y']):
            for dx, dy in [(stride, 0), (0, stride), (stride, stride), (stride, -stride)]:
                j = lookup.get((int(x) + dx, int(y) + dy))
                if j is not None:
                    source.append(int(i))
                    target.append(j)
    return (np.asarray(source, dtype=int), np.asarray(target, dtype=int), roi_groups)

def neighbor_mean(values, source, target):
    total = np.zeros_like(values, dtype=np.float32)
    degree = np.zeros(values.shape[0], dtype=np.float32)
    np.add.at(total, source, values[target])
    np.add.at(total, target, values[source])
    np.add.at(degree, source, 1)
    np.add.at(degree, target, 1)
    isolated = degree == 0
    degree[isolated] = 1
    total /= degree[:, None]
    total[isolated] = values[isolated]
    return total

def relabel_by_tissue(labels, frame):
    profile = frame.assign(_label=labels).groupby('_label')[TISSUE_COLUMNS].mean()
    profile['_dominant'] = np.argmax(profile.to_numpy(), axis=1)
    order = profile.sort_values(['_dominant', 'tumour_fraction', 'inflamed_stroma_fraction', 'tumour_associated_stroma_fraction'], ascending=[True, False, False, False]).index.tolist()
    mapping = {old: new for new, old in enumerate(order)}
    return (np.asarray([mapping[value] for value in labels], dtype=int), mapping)

def profile_table(frame, label_column, label_name):
    rows = []
    for label, group in frame.groupby(label_column, sort=True):
        tissue_means = group[TISSUE_COLUMNS].mean().to_dict()
        dominant = max(tissue_means, key=tissue_means.get).replace('_fraction', '').replace('_', ' ')
        rows.append({label_name: int(label), 'patch_count': int(len(group)), 'case_count': int(group['case_id'].nunique()), 'roi_count': int(group[['case_id', 'roi_id']].drop_duplicates().shape[0]), **{column: float(group[column].mean()) for column in COMPOSITION_COLUMNS}, 'dominant_tissue': dominant})
    return pd.DataFrame(rows)

def normalized_histogram(frame, label_column, prefix, labels, index_columns=('case_id',)):
    table = pd.crosstab([frame[column] for column in index_columns], frame[label_column], normalize='index').reindex(columns=labels, fill_value=0.0)
    table.columns = [f'{prefix}_{int(value):02d}' for value in labels]
    return table

def edge_pair_counts(labels, source, target, k):
    left = np.minimum(labels[source], labels[target])
    right = np.maximum(labels[source], labels[target])
    codes = left * k + right
    counts = np.bincount(codes, minlength=k * k).reshape(k, k)
    return counts[np.triu_indices(k)]

def permutation_subgroup_test(frame, group_column, label_column, permutations, seed):
    labelled = frame.dropna(subset=[label_column]).copy()
    if labelled[group_column].nunique() < 2 or labelled[label_column].nunique() < 2:
        return {'statistic': np.nan, 'p_value': np.nan, 'permutations': 0}
    observed = pd.crosstab(labelled[group_column], labelled[label_column])
    statistic = stats.chi2_contingency(observed, correction=False)[0]
    rng = np.random.default_rng(seed)
    groups = labelled[group_column].to_numpy()
    values = labelled[label_column].to_numpy()
    exceed = 0
    for _ in range(permutations):
        candidate = stats.chi2_contingency(pd.crosstab(groups, rng.permutation(values)), correction=False)[0]
        exceed += candidate >= statistic
    return {'statistic': float(statistic), 'p_value': float((exceed + 1) / (permutations + 1)), 'permutations': int(permutations)}

def permanova_two_group(matrix, labels, permutations, seed):
    matrix = np.asarray(matrix, dtype=float)
    labels = np.asarray(labels)
    distance = np.sqrt(np.maximum(((matrix[:, None, :] - matrix[None, :, :]) ** 2).sum(axis=2), 0))
    n = len(labels)
    centered = np.eye(n) - np.ones((n, n)) / n
    gram = -0.5 * centered @ distance ** 2 @ centered

    def pseudo_f(group_labels):
        unique = np.unique(group_labels)
        between = sum((gram[np.ix_(group_labels == value, group_labels == value)].sum() / np.sum(group_labels == value) for value in unique))
        total = np.trace(gram)
        within = max(total - between, np.finfo(float).eps)
        return between / (len(unique) - 1) / (within / (n - len(unique)))
    observed = pseudo_f(labels)
    rng = np.random.default_rng(seed)
    exceed = sum((pseudo_f(rng.permutation(labels)) >= observed for _ in range(permutations)))
    return (observed, (exceed + 1) / (permutations + 1))

def save_figure(fig, stem):
    png = FIGURES / f'{stem}.png'
    pdf = FIGURES / f'{stem}.pdf'
    fig.savefig(png, bbox_inches='tight', facecolor='white')
    fig.savefig(pdf, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    return png


## Feature and spatial representation


In [ ]:
preprocess_cache = CHECKPOINTS / '01_spatial_context.joblib'
preprocess_config = {'version': ANALYSIS_VERSION, 'quick_test': QUICK_TEST, 'seed': SEED, 'pca_dim': PCA_DIM, 'graph_steps': GRAPH_STEPS, 'graph_alpha': GRAPH_ALPHA}
if preprocess_cache.exists() and (not FORCE):
    cached = joblib.load(preprocess_cache)
    if cached.get('config') == preprocess_config:
        patches = cached['patches']
        context = cached['context']
        pca_scores = cached['pca_scores']
        source = cached['source']
        target = cached['target']
        roi_groups = cached['roi_groups']
        feature_columns = cached['feature_columns']
        pca_variance = cached['pca_variance']
    else:
        preprocess_cache.unlink()
if not preprocess_cache.exists():
    case_labels = pd.read_parquet(FEATURE_DATA / 'case_level_features_final.parquet', columns=['case_id', 'pcr_label'])
    case_paths = sorted(CASES.glob('*/features.parquet'))
    assert case_paths, f'No feature files found in {CASES}'
    if QUICK_TEST:
        selected_cases = case_labels['case_id'].sort_values().head(8).tolist()
        case_paths = [path for path in case_paths if path.parent.name in selected_cases]
    first_columns = pd.read_parquet(case_paths[0]).columns.tolist()
    feature_columns = sorted((column for column in first_columns if column.startswith('feature_')))
    assert len(feature_columns) == 768, f'Expected 768 features, found {len(feature_columns)}'
    patches = pd.concat([pd.read_parquet(path, columns=META_COLUMNS + feature_columns) for path in case_paths], ignore_index=True).sort_values(['case_id', 'roi_id', 'patch_y', 'patch_x']).reset_index(drop=True)
    assert not patches['patch_id'].duplicated().any(), 'Duplicate patch IDs detected'
    feature_matrix = patches[feature_columns].to_numpy(dtype=np.float32)
    assert np.isfinite(feature_matrix).all(), 'Non-finite feature values detected'
    standardized = StandardScaler().fit_transform(feature_matrix).astype(np.float32)
    pca = PCA(n_components=min(PCA_DIM, len(patches) - 1), svd_solver='randomized', random_state=SEED)
    pca_scores = pca.fit_transform(standardized).astype(np.float32)
    pca_variance = pd.DataFrame({'component': np.arange(1, pca.n_components_ + 1), 'explained_variance_ratio': pca.explained_variance_ratio_, 'cumulative_explained_variance': np.cumsum(pca.explained_variance_ratio_)})
    composition = patches[COMPOSITION_COLUMNS].fillna(0).to_numpy(dtype=np.float32)
    node_values = StandardScaler().fit_transform(np.column_stack([pca_scores, composition])).astype(np.float32)
    source, target, roi_groups = spatial_pairs(patches)
    levels = [node_values]
    current = node_values
    for _ in range(GRAPH_STEPS):
        current = ((1 - GRAPH_ALPHA) * current + GRAPH_ALPHA * neighbor_mean(current, source, target)).astype(np.float32)
        levels.append(current)
    context = StandardScaler().fit_transform(np.column_stack(levels)).astype(np.float32)
    joblib.dump({'config': preprocess_config, 'patches': patches[META_COLUMNS], 'context': context, 'pca_scores': pca_scores, 'source': source, 'target': target, 'roi_groups': roi_groups, 'feature_columns': feature_columns, 'pca_variance': pca_variance}, preprocess_cache, compress=3)
pca_variance.to_csv(DATA / 'pca_explained_variance.csv', index=False)


## Motif selection and coarse hierarchy


In [ ]:
cluster_cache = CHECKPOINTS / '02_hierarchical_clusters.joblib'
cluster_config = {'version': ANALYSIS_VERSION, 'fine_k_candidates': FINE_K_CANDIDATES, 'coarse_k_candidates': COARSE_K_CANDIDATES, 'stability_runs': PATCH_STABILITY_RUNS, 'seed': SEED}
if cluster_cache.exists() and (not FORCE):
    clustered = joblib.load(cluster_cache)
    if clustered.get('config') != cluster_config:
        cluster_cache.unlink()
if not cluster_cache.exists():
    rng = np.random.default_rng(SEED)
    silhouette_indices = rng.choice(len(context), size=min(5000, len(context)), replace=False)
    fine_rows = []
    seed_rows = []
    candidate_labels = {}
    for k in FINE_K_CANDIDATES:
        runs = []
        inertias = []
        for run in range(PATCH_STABILITY_RUNS):
            seed = SEED + 1000 * k + run
            model = MiniBatchKMeans(n_clusters=k, batch_size=2048, n_init=3, max_iter=300, reassignment_ratio=0.005, random_state=seed)
            labels = model.fit_predict(context)
            runs.append(labels)
            inertias.append(float(model.inertia_))
        ari = np.eye(len(runs))
        fmi = np.eye(len(runs))
        for left, right in combinations(range(len(runs)), 2):
            ari[left, right] = ari[right, left] = adjusted_rand_score(runs[left], runs[right])
            fmi[left, right] = fmi[right, left] = fowlkes_mallows_score(runs[left], runs[right])
        medoid = int(np.argmax((ari.sum(axis=1) - 1) / max(len(runs) - 1, 1)))
        labels = runs[medoid]
        counts = np.bincount(labels, minlength=k)
        silhouette = silhouette_score(context[silhouette_indices], labels[silhouette_indices])
        median_ari = float(np.median(ari[np.triu_indices(len(runs), 1)])) if len(runs) > 1 else 1.0
        median_fmi = float(np.median(fmi[np.triu_indices(len(runs), 1)])) if len(runs) > 1 else 1.0
        balance = normalized_entropy(counts)
        eligible = counts.min() / len(labels) >= MIN_PATCH_CLUSTER_FRACTION
        selection_score = 0.4 * median_ari + 0.25 * median_fmi + 0.25 * ((silhouette + 1) / 2) + 0.1 * balance
        fine_rows.append({'k': k, 'median_ari': median_ari, 'median_fmi': median_fmi, 'silhouette': float(silhouette), 'balance': balance, 'minimum_cluster_fraction': float(counts.min() / len(labels)), 'selection_score': float(selection_score), 'eligible': bool(eligible), 'medoid_seed': SEED + 1000 * k + medoid})
        candidate_labels[k] = labels
        for run, (seed, inertia) in enumerate(zip(range(SEED + 1000 * k, SEED + 1000 * k + PATCH_STABILITY_RUNS), inertias)):
            seed_rows.append({'k': k, 'run': run + 1, 'seed': seed, 'ari_to_medoid': adjusted_rand_score(labels, runs[run]), 'fmi_to_medoid': fowlkes_mallows_score(labels, runs[run]), 'inertia': inertia})
    fine_selection = pd.DataFrame(fine_rows)
    eligible = fine_selection[fine_selection['eligible']]
    selected_fine_k = int((eligible if not eligible.empty else fine_selection).sort_values(['selection_score', 'median_ari', 'silhouette'], ascending=False).iloc[0]['k'])
    fine_zero, _ = relabel_by_tissue(candidate_labels[selected_fine_k], patches)
    adjacency = np.zeros((selected_fine_k, selected_fine_k), dtype=float)
    for left, right in zip(fine_zero[source], fine_zero[target]):
        adjacency[left, right] += 1
        adjacency[right, left] += 1
    adjacency_normalized = adjacency / np.maximum(adjacency.sum(axis=1, keepdims=True), 1)
    centroids = np.vstack([context[fine_zero == value].mean(axis=0) for value in range(selected_fine_k)])
    coarse_input = StandardScaler().fit_transform(np.column_stack([centroids, adjacency_normalized]))
    coarse_rows = []
    coarse_candidates = {}
    for k in [value for value in COARSE_K_CANDIDATES if value < selected_fine_k]:
        labels = AgglomerativeClustering(n_clusters=k, linkage='ward').fit_predict(coarse_input)
        counts = np.bincount(labels, minlength=k)
        silhouette = silhouette_score(coarse_input, labels)
        balance = normalized_entropy(counts)
        eligible = counts.min() >= 2
        selection_score = 0.75 * ((silhouette + 1) / 2) + 0.25 * balance
        coarse_rows.append({'k': k, 'silhouette': float(silhouette), 'balance': balance, 'minimum_motif_count': int(counts.min()), 'selection_score': float(selection_score), 'eligible': bool(eligible)})
        coarse_candidates[k] = labels
    coarse_selection = pd.DataFrame(coarse_rows)
    eligible = coarse_selection[coarse_selection['eligible']]
    selected_coarse_k = int((eligible if not eligible.empty else coarse_selection).sort_values(['selection_score', 'silhouette'], ascending=False).iloc[0]['k'])
    coarse_zero = coarse_candidates[selected_coarse_k]
    fine_frame = patches.assign(_fine=fine_zero)
    coarse_profile = fine_frame.assign(_coarse=[coarse_zero[value] for value in fine_zero]).groupby('_coarse')[TISSUE_COLUMNS].mean()
    coarse_profile['_dominant'] = np.argmax(coarse_profile.to_numpy(), axis=1)
    coarse_order = coarse_profile.sort_values(['_dominant', 'tumour_fraction'], ascending=[True, False]).index.tolist()
    coarse_mapping = {old: new for new, old in enumerate(coarse_order)}
    coarse_zero = np.asarray([coarse_mapping[value] for value in coarse_zero], dtype=int)
    fine_labels = fine_zero + 1
    coarse_labels = np.asarray([coarse_zero[value] + 1 for value in fine_zero], dtype=int)
    assignments = patches.copy()
    assignments['fine_motif'] = fine_labels
    assignments['coarse_community'] = coarse_labels
    fine_profiles = profile_table(assignments, 'fine_motif', 'fine_motif')
    coarse_profiles = profile_table(assignments, 'coarse_community', 'coarse_community')
    fine_to_coarse = pd.DataFrame({'fine_motif': np.arange(1, selected_fine_k + 1), 'coarse_community': coarse_zero + 1})
    distances = np.linalg.norm(context - centroids[fine_zero], axis=1)
    representatives = assignments.assign(distance_to_centroid=distances).sort_values(['fine_motif', 'distance_to_centroid']).groupby('fine_motif', as_index=False).head(8)
    representative_columns = ['fine_motif', 'coarse_community', 'case_id', 'roi_id', 'patch_id', 'patch_x', 'patch_y', 'he_full_x', 'he_full_y', 'distance_to_centroid'] + TISSUE_COLUMNS
    representatives = representatives[representative_columns]
    clustered = {'config': cluster_config, 'fine_selection': fine_selection, 'seed_stability': pd.DataFrame(seed_rows), 'coarse_selection': coarse_selection, 'selected_fine_k': selected_fine_k, 'selected_coarse_k': selected_coarse_k, 'assignments': assignments, 'fine_profiles': fine_profiles, 'coarse_profiles': coarse_profiles, 'fine_to_coarse': fine_to_coarse, 'representatives': representatives, 'centroids': centroids, 'coarse_input': coarse_input}
    joblib.dump(clustered, cluster_cache, compress=3)
fine_selection = clustered['fine_selection']
seed_stability = clustered['seed_stability']
coarse_selection = clustered['coarse_selection']
selected_fine_k = clustered['selected_fine_k']
selected_coarse_k = clustered['selected_coarse_k']
assignments = clustered['assignments']
fine_profiles = clustered['fine_profiles']
coarse_profiles = clustered['coarse_profiles']
fine_to_coarse = clustered['fine_to_coarse']
representatives = clustered['representatives']
coarse_input = clustered['coarse_input']
assignments.to_parquet(DATA / 'patch_community_assignments.parquet', index=False)
fine_selection.to_csv(DATA / 'fine_k_selection.csv', index=False)
seed_stability.to_csv(DATA / 'fine_seed_stability.csv', index=False)
seed_stability[seed_stability['k'] == selected_fine_k].to_csv(DATA / 'clustering_stability.csv', index=False)
coarse_selection.to_csv(DATA / 'coarse_k_selection.csv', index=False)
fine_profiles.to_csv(DATA / 'fine_motif_profiles.csv', index=False)
coarse_profiles.to_csv(DATA / 'coarse_community_profiles.csv', index=False)
fine_to_coarse.to_csv(DATA / 'fine_to_coarse_mapping.csv', index=False)
representatives.to_csv(DATA / 'representative_patch_coordinates.csv', index=False)


## Spatial interactions and ROI heterogeneity


In [ ]:
spatial_cache = CHECKPOINTS / '03_spatial_statistics.joblib'
spatial_config = {'version': ANALYSIS_VERSION, 'fine_k': selected_fine_k, 'coarse_k': selected_coarse_k, 'permutations': SPATIAL_PERMUTATIONS, 'seed': SEED}
if spatial_cache.exists() and (not FORCE):
    spatial = joblib.load(spatial_cache)
    if spatial.get('config') != spatial_config:
        spatial_cache.unlink()
if not spatial_cache.exists():
    fine_zero = assignments['fine_motif'].to_numpy(dtype=int) - 1
    coarse_zero_patch = assignments['coarse_community'].to_numpy(dtype=int) - 1
    observed = edge_pair_counts(fine_zero, source, target, selected_fine_k)
    permutations = np.zeros((SPATIAL_PERMUTATIONS, len(observed)), dtype=np.float32)
    rng = np.random.default_rng(SEED + 4000)
    for iteration in range(SPATIAL_PERMUTATIONS):
        permuted = fine_zero.copy()
        for indices in roi_groups.values():
            permuted[indices] = rng.permutation(permuted[indices])
        permutations[iteration] = edge_pair_counts(permuted, source, target, selected_fine_k)
    expected = permutations.mean(axis=0)
    standard_deviation = permutations.std(axis=0, ddof=1)
    differences = observed - expected
    p_values = (np.sum(np.abs(permutations - expected) >= np.abs(differences), axis=0) + 1) / (SPATIAL_PERMUTATIONS + 1)
    upper = np.triu_indices(selected_fine_k)
    spatial_interactions = pd.DataFrame({'motif_a': upper[0] + 1, 'motif_b': upper[1] + 1, 'observed_edges': observed, 'expected_edges': expected, 'fold_enrichment': (observed + 0.5) / (expected + 0.5), 'z_score': differences / np.where(standard_deviation > 0, standard_deviation, np.nan), 'p_value': p_values})
    spatial_interactions['fdr_bh'] = bh_fdr(spatial_interactions['p_value'])
    edge_frame = pd.DataFrame({'source': source, 'target': target, 'case_id': assignments.iloc[source]['case_id'].to_numpy(), 'roi_id': assignments.iloc[source]['roi_id'].to_numpy()})
    roi_rows = []
    for key, indices in roi_groups.items():
        case_id, roi_id = key
        group = assignments.loc[indices]
        roi_edges = edge_frame[(edge_frame['case_id'] == case_id) & (edge_frame['roi_id'] == roi_id)]
        edge_source = roi_edges['source'].to_numpy(dtype=int)
        edge_target = roi_edges['target'].to_numpy(dtype=int)
        fine_counts = np.bincount(fine_zero[indices], minlength=selected_fine_k)
        coarse_counts = np.bincount(coarse_zero_patch[indices], minlength=selected_coarse_k)
        roi_rows.append({'case_id': case_id, 'roi_id': roi_id, 'patch_count': int(len(indices)), 'fine_motif_richness': int(np.sum(fine_counts > 0)), 'fine_entropy': normalized_entropy(fine_counts), 'coarse_entropy': normalized_entropy(coarse_counts), 'fine_homotypic_edge_fraction': float(np.mean(fine_zero[edge_source] == fine_zero[edge_target])) if len(edge_source) else np.nan, 'coarse_homotypic_edge_fraction': float(np.mean(coarse_zero_patch[edge_source] == coarse_zero_patch[edge_target])) if len(edge_source) else np.nan, 'fine_interface_edge_fraction': float(np.mean(fine_zero[edge_source] != fine_zero[edge_target])) if len(edge_source) else np.nan, **{column: float(group[column].mean()) for column in COMPOSITION_COLUMNS}, 'mif_signal_fraction': float(group['mif_signal_fraction'].mean())})
    roi_heterogeneity = pd.DataFrame(roi_rows)
    spatial = {'config': spatial_config, 'spatial_interactions': spatial_interactions, 'roi_heterogeneity': roi_heterogeneity}
    joblib.dump(spatial, spatial_cache, compress=3)
spatial_interactions = spatial['spatial_interactions']
roi_heterogeneity = spatial['roi_heterogeneity']
spatial_interactions.to_csv(DATA / 'spatial_motif_interactions.csv', index=False)
roi_heterogeneity.to_csv(DATA / 'roi_spatial_heterogeneity.csv', index=False)


## Case profiles and exploratory pCR tests


In [ ]:
patient_cache = CHECKPOINTS / '04_patient_analysis.joblib'
patient_config = {'version': ANALYSIS_VERSION, 'fine_k': selected_fine_k, 'coarse_k': selected_coarse_k, 'bootstraps': PATIENT_BOOTSTRAPS, 'pcr_permutations': PCR_PERMUTATIONS, 'seed': SEED}
if patient_cache.exists() and (not FORCE):
    patient = joblib.load(patient_cache)
    if patient.get('config') != patient_config:
        patient_cache.unlink()
if not patient_cache.exists():
    case_labels = pd.read_parquet(FEATURE_DATA / 'case_level_features_final.parquet', columns=['case_id', 'pcr_label'])
    fine_hist = normalized_histogram(assignments, 'fine_motif', 'fine_motif', range(1, selected_fine_k + 1))
    coarse_hist = normalized_histogram(assignments, 'coarse_community', 'coarse_community', range(1, selected_coarse_k + 1))
    case_histograms = fine_hist.join(coarse_hist).reset_index()
    roi_summary = roi_heterogeneity.groupby('case_id').agg(roi_count=('roi_id', 'nunique'), patch_count=('patch_count', 'sum'), fine_entropy_mean=('fine_entropy', 'mean'), fine_entropy_sd=('fine_entropy', 'std'), coarse_entropy_mean=('coarse_entropy', 'mean'), fine_homotypic_edge_fraction_mean=('fine_homotypic_edge_fraction', 'mean'), coarse_homotypic_edge_fraction_mean=('coarse_homotypic_edge_fraction', 'mean'), fine_interface_edge_fraction_mean=('fine_interface_edge_fraction', 'mean'), tumour_fraction_mean=('tumour_fraction', 'mean'), tumour_associated_stroma_fraction_mean=('tumour_associated_stroma_fraction', 'mean'), inflamed_stroma_fraction_mean=('inflamed_stroma_fraction', 'mean'), other_fraction_mean=('other_fraction', 'mean'), mif_signal_fraction_mean=('mif_signal_fraction', 'mean')).reset_index()
    registration = assignments.groupby('case_id')['registration_status'].agg(lambda values: values.mode().iloc[0]).reset_index()
    case_profiles = case_histograms.merge(roi_summary, on='case_id', how='left').merge(registration, on='case_id', how='left').merge(case_labels, on='case_id', how='left')
    pattern_columns = [column for column in case_profiles if column.startswith(('fine_motif_', 'coarse_community_'))]
    representation = np.sqrt(np.clip(case_profiles[pattern_columns].to_numpy(dtype=float), 0, None))
    similarity_values = cosine_similarity(representation)
    np.fill_diagonal(similarity_values, 1.0)
    distance = np.clip(1 - similarity_values, 0, 2)
    np.fill_diagonal(distance, 0)
    roi_counts = normalized_histogram(assignments, 'fine_motif', 'fine_motif', range(1, selected_fine_k + 1), index_columns=('case_id', 'roi_id')).join(normalized_histogram(assignments, 'coarse_community', 'coarse_community', range(1, selected_coarse_k + 1), index_columns=('case_id', 'roi_id')))
    candidate_rows = []
    candidate_labels = {}
    rng = np.random.default_rng(SEED + 5000)
    for k in range(2, min(4, len(case_profiles) - 1) + 1):
        labels = AgglomerativeClustering(n_clusters=k, metric='precomputed', linkage='average').fit_predict(distance)
        counts = np.bincount(labels, minlength=k)
        silhouette = silhouette_score(distance, labels, metric='precomputed')
        bootstrap_ari = []
        for _ in range(PATIENT_BOOTSTRAPS):
            bootstrap_rows = []
            for case_id in case_profiles['case_id']:
                roi_matrix = roi_counts.loc[case_id]
                if isinstance(roi_matrix, pd.Series):
                    roi_matrix = roi_matrix.to_frame().T
                sampled = roi_matrix.iloc[rng.integers(0, len(roi_matrix), size=len(roi_matrix))]
                bootstrap_rows.append(sampled.mean(axis=0).to_numpy(dtype=float))
            bootstrap_representation = np.sqrt(np.clip(np.vstack(bootstrap_rows), 0, None))
            bootstrap_similarity = cosine_similarity(bootstrap_representation)
            bootstrap_distance = np.clip(1 - bootstrap_similarity, 0, 2)
            np.fill_diagonal(bootstrap_distance, 0)
            bootstrap_labels = AgglomerativeClustering(n_clusters=k, metric='precomputed', linkage='average').fit_predict(bootstrap_distance)
            bootstrap_ari.append(adjusted_rand_score(labels, bootstrap_labels))
        balance = normalized_entropy(counts)
        median_bootstrap_ari = float(np.median(bootstrap_ari))
        eligible = counts.min() >= MIN_PATIENT_GROUP_SIZE
        selection_score = 0.45 * ((silhouette + 1) / 2) + 0.45 * median_bootstrap_ari + 0.1 * balance
        candidate_rows.append({'k': k, 'silhouette': float(silhouette), 'bootstrap_ari_median': median_bootstrap_ari, 'bootstrap_ari_p10': float(np.quantile(bootstrap_ari, 0.1)), 'balance': balance, 'minimum_group_size': int(counts.min()), 'selection_score': float(selection_score), 'eligible': bool(eligible)})
        candidate_labels[k] = labels
    patient_k_selection = pd.DataFrame(candidate_rows)
    eligible = patient_k_selection[patient_k_selection['eligible']]
    selected_patient_k = int((eligible if not eligible.empty else patient_k_selection).sort_values(['selection_score', 'bootstrap_ari_median', 'silhouette'], ascending=False).iloc[0]['k'])
    subgroup_zero = candidate_labels[selected_patient_k]
    size_order = pd.Series(subgroup_zero).value_counts().sort_values(ascending=False).index.tolist()
    subgroup_map = {old: new + 1 for new, old in enumerate(size_order)}
    case_profiles['patient_subgroup'] = [subgroup_map[value] for value in subgroup_zero]
    population_similarity = pd.DataFrame(similarity_values, index=case_profiles['case_id'].astype(str), columns=case_profiles['case_id'].astype(str))
    metric_columns = pattern_columns + ['fine_entropy_mean', 'coarse_entropy_mean', 'fine_homotypic_edge_fraction_mean', 'coarse_homotypic_edge_fraction_mean', 'fine_interface_edge_fraction_mean', 'tumour_fraction_mean', 'tumour_associated_stroma_fraction_mean', 'inflamed_stroma_fraction_mean', 'other_fraction_mean', 'mif_signal_fraction_mean']
    association_rows = []
    for analysis_set, subset in {'all_labelled_cases': case_profiles.dropna(subset=['pcr_label']), 'cases_with_3_feature_rois': case_profiles.dropna(subset=['pcr_label']).query('roi_count >= 3')}.items():
        for column in metric_columns:
            pcr = subset.loc[subset['pcr_label'] == 1, column].dropna().to_numpy(dtype=float)
            non_pcr = subset.loc[subset['pcr_label'] == 0, column].dropna().to_numpy(dtype=float)
            if len(pcr) < 2 or len(non_pcr) < 2:
                continue
            result = stats.mannwhitneyu(pcr, non_pcr, alternative='two-sided')
            hodges_lehmann = float(np.median(pcr[:, None] - non_pcr[None, :]))
            association_rows.append({'analysis_set': analysis_set, 'metric': column, 'pcr_n': len(pcr), 'non_pcr_n': len(non_pcr), 'pcr_median': float(np.median(pcr)), 'non_pcr_median': float(np.median(non_pcr)), 'hodges_lehmann_difference': hodges_lehmann, 'rank_biserial': float(2 * result.statistic / (len(pcr) * len(non_pcr)) - 1), 'p_value': float(result.pvalue)})
    pcr_associations = pd.DataFrame(association_rows)
    if not pcr_associations.empty:
        pcr_associations['fdr_bh'] = pcr_associations.groupby('analysis_set')['p_value'].transform(bh_fdr)
        pcr_associations = pcr_associations.sort_values(['analysis_set', 'fdr_bh', 'p_value']).reset_index(drop=True)
    labelled = case_profiles.dropna(subset=['pcr_label']).copy()
    fine_columns = [column for column in pattern_columns if column.startswith('fine_motif_')]
    permanova_statistic, permanova_p = permanova_two_group(np.sqrt(labelled[fine_columns].to_numpy(dtype=float)), labelled['pcr_label'].to_numpy(), PCR_PERMUTATIONS, SEED + 6000)
    subgroup_test = permutation_subgroup_test(case_profiles, 'patient_subgroup', 'pcr_label', PCR_PERMUTATIONS, SEED + 7000)
    global_tests = pd.DataFrame([{'test': 'PERMANOVA on Hellinger-transformed fine-motif composition', 'statistic': permanova_statistic, 'p_value': permanova_p, 'permutations': PCR_PERMUTATIONS}, {'test': 'Permutation chi-square: patient subgroup versus pCR', **subgroup_test}])
    technical_metrics = ['patch_count', 'roi_count', 'tumour_fraction_mean', 'tumour_associated_stroma_fraction_mean', 'inflamed_stroma_fraction_mean', 'other_fraction_mean', 'mif_signal_fraction_mean']
    technical_rows = []
    for column in technical_metrics:
        groups = [group[column].dropna().to_numpy(dtype=float) for _, group in case_profiles.groupby('patient_subgroup')]
        if len(groups) == 2:
            result = stats.mannwhitneyu(groups[0], groups[1], alternative='two-sided')
            effect = 2 * result.statistic / (len(groups[0]) * len(groups[1])) - 1
        else:
            result = stats.kruskal(*groups)
            effect = np.nan
        technical_rows.append({'metric': column, 'test': 'Mann-Whitney U' if len(groups) == 2 else 'Kruskal-Wallis', 'statistic': float(result.statistic), 'effect_rank_biserial': float(effect) if np.isfinite(effect) else np.nan, 'p_value': float(result.pvalue)})
    technical_confounding = pd.DataFrame(technical_rows)
    technical_confounding['fdr_bh'] = bh_fdr(technical_confounding['p_value'])
    patient = {'config': patient_config, 'case_profiles': case_profiles, 'population_similarity': population_similarity, 'patient_k_selection': patient_k_selection, 'selected_patient_k': selected_patient_k, 'pcr_associations': pcr_associations, 'global_tests': global_tests, 'technical_confounding': technical_confounding}
    joblib.dump(patient, patient_cache, compress=3)
case_profiles = patient['case_profiles']
population_similarity = patient['population_similarity']
patient_k_selection = patient['patient_k_selection']
selected_patient_k = patient['selected_patient_k']
pcr_associations = patient['pcr_associations']
global_tests = patient['global_tests']
technical_confounding = patient['technical_confounding']
case_profiles.to_csv(DATA / 'case_spatial_profiles.csv', index=False)
population_similarity.to_csv(DATA / 'population_similarity.csv')
patient_k_selection.to_csv(DATA / 'patient_subgroup_k_selection.csv', index=False)
case_profiles[['case_id', 'pcr_label', 'patient_subgroup', 'patch_count', 'roi_count', 'registration_status']].to_csv(DATA / 'patient_subgroups.csv', index=False)
pcr_associations.to_csv(DATA / 'pcr_case_level_associations.csv', index=False)
pcr_associations[pcr_associations['analysis_set'] == 'all_labelled_cases'].to_csv(DATA / 'pcr_pattern_associations.csv', index=False)
global_tests.to_csv(DATA / 'pcr_global_tests.csv', index=False)
technical_confounding.to_csv(DATA / 'technical_confounding_tests.csv', index=False)


## Report figures


In [ ]:
fine_colors = [PALETTE[index % len(PALETTE)] for index in range(selected_fine_k)]
coarse_colors = [PALETTE[(index * 3 + 1) % len(PALETTE)] for index in range(selected_coarse_k)]
subgroup_colors = [PALETTE[index * 4 % len(PALETTE)] for index in range(selected_patient_k)]
figure_paths = []
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
axes[0].plot(fine_selection['k'], fine_selection['median_ari'], 'o-', color='#58508D', label='Median ARI')
axes[0].plot(fine_selection['k'], fine_selection['median_fmi'], 's--', color='#FF6361', label='Median FMI')
axes[0].axvline(selected_fine_k, color='#FFA600', linestyle=':', linewidth=2)
axes[0].set_title('Fine-motif stability')
axes[0].set_xlabel('Number of motifs')
axes[0].set_ylim(0, 1)
axes[0].legend(frameon=False)
axes[1].scatter(fine_selection['silhouette'], fine_selection['selection_score'], s=500 * fine_selection['minimum_cluster_fraction'] + 25, c=fine_selection['k'], cmap='viridis', edgecolor='white')
for row in fine_selection.itertuples(index=False):
    axes[1].annotate(f'k={row.k}', (row.silhouette, row.selection_score), xytext=(4, 4), textcoords='offset points', fontsize=8)
axes[1].set_title('Fine-motif model selection')
axes[1].set_xlabel('Silhouette')
axes[1].set_ylabel('Composite score')
axes[2].bar(coarse_selection['k'].astype(str), coarse_selection['silhouette'], color=[coarse_colors[(int(k) - 1) % len(coarse_colors)] for k in coarse_selection['k']])
axes[2].axhline(coarse_selection.loc[coarse_selection['k'] == selected_coarse_k, 'silhouette'].iloc[0], color='#222222', linestyle=':')
axes[2].set_title('Coarse-community selection')
axes[2].set_xlabel('Number of communities')
axes[2].set_ylabel('Silhouette')
fig.suptitle('Cluster number selection and reproducibility', fontsize=16, weight='bold')
fig.tight_layout()
figure_paths.append(save_figure(fig, '02_cluster_selection_stability'))
fig = plt.figure(figsize=(14, 7))
grid = fig.add_gridspec(1, 2, width_ratios=[1.05, 1.4])
ax = fig.add_subplot(grid[0, 0])
coarse_groups = fine_to_coarse.groupby('coarse_community')['fine_motif'].apply(list)
y_positions = {motif: selected_fine_k - motif for motif in range(1, selected_fine_k + 1)}
coarse_y = {community: np.mean([y_positions[motif] for motif in motifs]) for community, motifs in coarse_groups.items()}
for motif in range(1, selected_fine_k + 1):
    community = int(fine_to_coarse.loc[fine_to_coarse['fine_motif'] == motif, 'coarse_community'].iloc[0])
    y = y_positions[motif]
    yc = coarse_y[community]
    ax.add_patch(mpl_patches.FancyBboxPatch((0.03, y - 0.28), 0.28, 0.56, boxstyle='round,pad=0.02', facecolor=fine_colors[motif - 1], edgecolor='white'))
    ax.text(0.17, y, f'M{motif}', ha='center', va='center', color='white', weight='bold')
    ax.plot([0.31, 0.7], [y, yc], color=coarse_colors[community - 1], linewidth=3, alpha=0.5)
for community, yc in coarse_y.items():
    ax.add_patch(mpl_patches.FancyBboxPatch((0.7, yc - 0.38), 0.27, 0.76, boxstyle='round,pad=0.02', facecolor=coarse_colors[community - 1], edgecolor='white'))
    ax.text(0.835, yc, f'C{community}', ha='center', va='center', color='white', weight='bold')
ax.set_xlim(0, 1)
ax.set_ylim(-1, selected_fine_k)
ax.axis('off')
ax.set_title('Nested motif-to-community hierarchy')
ax2 = fig.add_subplot(grid[0, 1])
profile_matrix = fine_profiles.set_index('fine_motif')[TISSUE_COLUMNS]
sns.heatmap(profile_matrix, cmap='rocket_r', annot=True, fmt='.2f', linewidths=0.5, cbar_kws={'label': 'Mean fraction'}, ax=ax2)
ax2.set_xlabel('Segmentation-derived tissue composition')
ax2.set_ylabel('Fine motif')
ax2.set_title('Interpretable motif profiles')
fig.suptitle('Hierarchical spatial communities and tissue composition', fontsize=16, weight='bold')
fig.tight_layout()
figure_paths.append(save_figure(fig, '04_hierarchy_and_tissue_profiles'))
roi_rank = roi_heterogeneity.sort_values(['fine_entropy', 'patch_count'], ascending=[False, False]).drop_duplicates('case_id').head(4)
fig = plt.figure(figsize=(15, 10))
for panel, row in enumerate(roi_rank.itertuples(index=False), start=1):
    ax = fig.add_subplot(2, 2, panel, projection='3d')
    group = assignments[(assignments['case_id'] == row.case_id) & (assignments['roi_id'] == row.roi_id)]
    x_values = group['patch_x'].to_numpy(dtype=float)
    y_values = group['patch_y'].to_numpy(dtype=float)
    x_values = (x_values - x_values.min()) / max(np.ptp(x_values), 1)
    y_values = (y_values - y_values.min()) / max(np.ptp(y_values), 1)
    z_values = group['coarse_community'].to_numpy(dtype=float) * 0.12
    colors = [fine_colors[value - 1] for value in group['fine_motif'].to_numpy(dtype=int)]
    ax.scatter(x_values, y_values, z_values, c=colors, marker='s', s=55, alpha=0.86, edgecolor='white', linewidth=0.25)
    ax.set_title(f'{row.case_id} · {row.roi_id}\nentropy={row.fine_entropy:.2f}')
    ax.set_xlabel('ROI x')
    ax.set_ylabel('ROI y')
    ax.set_zlabel('Community layer')
    ax.set_zticks([])
    ax.view_init(elev=28, azim=-58)
    ax.grid(False)
fig.suptitle('Three-dimensional spatial motif landscapes', fontsize=17, weight='bold')
fig.tight_layout()
figure_paths.append(save_figure(fig, '06_3D_spatial_landscapes'))
population_distance = np.sqrt(np.clip(1 - population_similarity.to_numpy(), 0, 2))
np.fill_diagonal(population_distance, 0)
order = leaves_list(linkage(squareform(population_distance, checks=False), method='average'))
ordered_cases = case_profiles.iloc[order]['case_id'].astype(str).tolist()
ordered_similarity = population_similarity.loc[ordered_cases, ordered_cases]
fig, ax = plt.subplots(figsize=(12.5, 10.5))
sns.heatmap(ordered_similarity, cmap='mako', vmin=0, vmax=1, xticklabels=ordered_cases, yticklabels=ordered_cases, cbar_kws={'label': 'Pattern similarity'}, ax=ax)
ax.tick_params(axis='x', labelrotation=90, labelsize=7)
ax.tick_params(axis='y', labelsize=7)
ax.set_title('Patient similarity from spatial motif composition')
fig.tight_layout()
figure_paths.append(save_figure(fig, '07_patient_similarity_heatmap'))
formal_effects = pcr_associations[pcr_associations['analysis_set'] == 'all_labelled_cases'].copy()
formal_effects = formal_effects.reindex(formal_effects['rank_biserial'].abs().sort_values(ascending=False).index).head(16).sort_values('rank_biserial')
fig, ax = plt.subplots(figsize=(10, 7))
effect_colors = ['#4C78A8' if value < 0 else '#E45756' for value in formal_effects['rank_biserial']]
ax.hlines(np.arange(len(formal_effects)), 0, formal_effects['rank_biserial'], color=effect_colors, linewidth=2)
ax.scatter(formal_effects['rank_biserial'], np.arange(len(formal_effects)), c=effect_colors, s=70, edgecolor='white', zorder=3)
ax.axvline(0, color='#333333', linewidth=0.8)
ax.set_yticks(np.arange(len(formal_effects)), formal_effects['metric'].str.replace('_', ' '))
ax.set_xlabel('Rank-biserial effect · positive means higher in pCR')
ax.set_title('Exploratory case-level pCR associations')
for index, row in enumerate(formal_effects.itertuples(index=False)):
    ax.text(row.rank_biserial, index + 0.2, f'p={row.p_value:.3f}', ha='center', fontsize=7, color='#555555')
fig.text(0.5, 0.01, 'Unadjusted p-values shown; see the output table for multiple-testing-adjusted values', ha='center', fontsize=8, color='#555555')
fig.tight_layout(rect=(0, 0.04, 1, 1))
figure_paths.append(save_figure(fig, '09_pCR_effect_size_forest'))


## Analysis manifest


In [ ]:
manifest = {'created_at': datetime.now().isoformat(timespec='seconds'), 'analysis': 'Enhanced hierarchical KRONOS2 patch community and patient stratification', 'config': CONFIG, 'case_count': int(assignments['case_id'].nunique()), 'feature_bearing_roi_count': int(assignments[['case_id', 'roi_id']].drop_duplicates().shape[0]), 'patch_count': int(len(assignments)), 'feature_dimension': 768, 'graph_edge_count': int(len(source)), 'selected_fine_motifs': int(selected_fine_k), 'selected_coarse_communities': int(selected_coarse_k), 'selected_patient_subgroups': int(selected_patient_k), 'label_usage': 'pCR labels were excluded from cluster discovery and used only post hoc.', 'limitations': ['Patch nodes are not cells; results are patch motifs and patch communities.', 'KRONOS2 dimensions are latent variables and are not individual protein identities.', 'Sampled ROIs do not provide complete whole-slide coverage.', 'Patient clusters require external validation.', 'pCR analyses are exploratory and use cases as independent units.'], 'software': {'python': sys.version.split()[0], 'platform': platform.platform(), 'numpy': np.__version__, 'pandas': pd.__version__}}
manifest_text = json.dumps(manifest, indent=2)
(OUTPUT / 'enhanced_spatial_community_manifest.json').write_text(manifest_text, encoding='utf-8')
(OUTPUT / 'spatial_community_graph_manifest.json').write_text(manifest_text, encoding='utf-8')
summary = pd.DataFrame([('Cases', manifest['case_count']), ('Feature-bearing ROIs', manifest['feature_bearing_roi_count']), ('Patches', manifest['patch_count']), ('Spatial edges', manifest['graph_edge_count']), ('Fine motifs', manifest['selected_fine_motifs']), ('Coarse communities', manifest['selected_coarse_communities']), ('Patient subgroups', manifest['selected_patient_subgroups'])], columns=['Measure', 'Value'])
display(summary)
display(fine_selection.round(4))
display(coarse_selection.round(4))
display(patient_k_selection.round(4))
display(global_tests.round(4))
display(technical_confounding.round(4))
display(pcr_associations[pcr_associations['analysis_set'] == 'all_labelled_cases'].head(15).round(4))
